[Setup instructions](./Summary.md#topic-notebooks)


In [ ]:
import numpy as np

from pathlib import Path
import sys

_shared_dir = Path.cwd()
if not (_shared_dir / "common.py").is_file():
    _shared_dir = _shared_dir / "02-regression"
if not (_shared_dir / "common.py").is_file():
    raise FileNotFoundError(
        "Run this notebook from the machine-learning or 02-regression folder."
    )
if str(_shared_dir) not in sys.path:
    sys.path.insert(0, str(_shared_dir))

from common import (
    base,
    df,
    df_train,
    df_val,
    y_train,
    y_val,
    train_linear_regression,
    rmse,
)


# Computing RMSE on validation data

## Validation

- Split data into **train**, **validation**, and **test** sets.
- Train the model only on the **training set**.
- Evaluate it on the **validation set**.
- Validation shows how the model performs on **unseen data**.

In [351]:
# Equivalent to: X_train = df_train[base].fillna(0).values
def prepare_X(df):
    df_num = df[base]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

### `prepare_X()`

- Selects the features in `base`.
- Replaces missing values with `0`.
- Converts the data to a NumPy array.
- Ensures train and validation data are prepared the same way.

In [352]:
# Train
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

# Validate
X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

np.float64(0.7616530991301561)

- Validation RMSE ≈ `0.762`
- Similar train and validation RMSE → model performs similarly on unseen data.
- Use **validation RMSE** to compare future model improvements.

## Feature Engineering

- **Feature engineering** = creating new features from existing data.
- Car `year` can be converted into **car age**.
- Dataset year: `2017`
- `age = 2017 - year`

In [353]:
2017 - df_train.year

0        9
1        5
2        1
3       26
4        0
        ..
7145     2
7146     2
7147     2
7148     3
7149     0
Name: year, Length: 7150, dtype: int64

### Add `age` to the model

- Add `age` as a new feature.
- Use the same preparation for train and validation data.
- Work on a copy to avoid changing the original dataframe.

In [354]:
def prepare_X(df):
    df = df.copy()

    df['age'] = 2017 - df.year
    features = base + ['age']

    df_num = df[features]
    df_num = df_num.fillna(0)

    return df_num.values

### Evaluate the new feature

In [355]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

np.float64(0.5172055461058324)

- Previous RMSE: `~0.76`
- With `age`: `~0.52`
- Lower RMSE → **significant improvement**
- Good features can strongly improve model performance.

## Categorical Variables

- Categorical variables represent **categories**, not continuous numbers.
- Examples: `make`, `fuel_type`, `transmission_type`.
- ML models need numbers → categories must be encoded.
- Common method: **One-Hot Encoding**.

### Example: Number of Doors

`number_of_doors` looks numerical, but values like `2`, `3`, `4` are categories.

One-hot encoding creates binary features:

- `num_doors_2`
- `num_doors_3`
- `num_doors_4`

In [356]:
for v in [2, 3, 4]:
    df_train['num_doors_%s' % v] = (
        df_train.number_of_doors == v
    ).astype(int)

### Add Doors to `prepare_X`

In [357]:
features = base.copy()

def prepare_X(df):
    df = df.copy()

    df['age'] = 2017 - df.year
    features.append('age')

    for v in [2, 3, 4]:
        feature = 'num_doors_%s' % v
        df[feature] = (df.number_of_doors == v).astype(int)
        features.append(feature)

    df_num = df[features].fillna(0)

    return df_num.values

- RMSE before doors: `~0.517`
- RMSE with doors: `~0.516`
- Very small improvement

### Adding Car Make

Use the most common car manufacturers.

In [358]:
makes = list(df.make.value_counts().head().index)

for v in makes:
    feature = 'make_%s' % v
    df[feature] = (df.make == v).astype(int)
    features.append(feature)

- RMSE improves to about `0.508`
- `make` is more useful than `number_of_doors`

### Add More Categorical Variables

In [359]:
categorical_variables = [
    'make',
    'engine_fuel_type',
    'transmission_type',
    'driven_wheels',
    'market_category',
    'vehicle_size',
    'vehicle_style'
]

categories = {}

for c in categorical_variables:
    categories[c] = list(df_train[c].value_counts().head().index)

In [360]:
for c, values in categories.items():
    for v in values:
        feature = '%s_%s' % (c, v)
        df[feature] = (df[c] == v).astype(int)
        features.append(feature)

### Problem

- After adding many categorical features, RMSE becomes `~41`.
- Model weights become extremely large.
- The model becomes **numerically unstable**.
- Next step: **regularization**.